<a href="https://colab.research.google.com/github/mf2056/F20AA/blob/main/DataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas textblob vaderSentiment scikit-learn

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving amazon_culture_data.csv to amazon_culture_data.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving youtube_amazon_employee.csv to youtube_amazon_employee.csv


In [ ]:
# Retrieving the Dataset
import pandas as pd

df1 = pd.read_csv('amazon_culture_data.csv')
df2 = pd.read_csv('youtube_amazon_employee.csv')
df1 = df1[["text"]]
df2 = df2[["text"]]
df = pd.concat([df1, df2], ignore_index=True)

df.head()

,text
0,I'm suspended for having my Medical Marijuana ...
1,Once again you enlighten US!\nKeep up the grea...
2,Don't ask me to work on my day off and you bet...
3,I'm a proud Amazon Prime member. I'm also atte...
4,Amazon employees are over worked with mandator...


In [ ]:
# Performing Basic Cleaning

comment_col = "text"
df = df.dropna(subset=[comment_col])  # Drop rows with missing comments
df[comment_col] = df[comment_col].astype(str).str.strip()

# Remove empty strings
df = df[df[comment_col] != ""]

# Remove duplicates
df = df.drop_duplicates(subset=[comment_col])

# Remove very short comments (less than 3 words)
word_count = df["text"].astype(str).str.split().str.len()
df = df[word_count >= 3]

df.shape

(5547, 1)

In [ ]:
# Remove spam comments

spam_keywords = [
    "subscribe", "giveaway", "click", "telegram", "whatsapp",
    "contact me", "dm me", "crypto", "bitcoin", "forex", "trading",
    "please like", "anyone watching in", "bit.ly", "goo.gl",
    "link in bio", "visit my site"
]

def remove_spam(text):
    text_lower = text.lower()
    return not any(word in text_lower for word in spam_keywords)

df = df[df[comment_col].apply(remove_spam)]
df.shape

(5524, 1)

In [ ]:
# Filtering relevant comments

workplace_keywords = [
    "employee", "worker", "staff", "associate", "warehouse",
    "leadership", "executive", "perks", "insurance",
    "fulfillment", "manager", "management", "boss",
    "hr", "supervisor", "shift", "overtime", "break", "pay",
    "salary", "wage", "benefits", "culture",
    "workload", "stress", "burnout", "treatment",
    "working", "job", "career", "fired", "hired", "workplace",
    "company", "office", "corporate", "hiring", "recruitment",
    "interview", "promotion", "resignation", "quit", "layoff",
]

def is_relevant(text):
    text_lower = text.lower()
    return any(word in text_lower for word in workplace_keywords)

df = df[df[comment_col].apply(is_relevant)]
df.shape

(3385, 1)

In [ ]:
# Labeling using TextBlob

from textblob import TextBlob

def textblob_polarity(text):
    return TextBlob(text).sentiment.polarity

tb_polarity = df["text"].apply(textblob_polarity)

def tb_to_label(p):
    if p > 0.05:
        return "positive"
    elif p < -0.05:
        return "negative"
    else:
        return "neutral"

df['tb_label'] = tb_polarity.apply(tb_to_label)
df['tb_label'].value_counts()

,count
tb_label,
positive,1470
neutral,1101
negative,814


In [ ]:
print("Top 5 Positive Comments:")
df[df["tb_label"] == "positive"]["text"].head(5).tolist()

Top 5 Positive Comments:


['Amazon treats its workers like dirt and then acts surprised when the workers want to unionize.?    They are getting what they asked for with the tactics they have used.',
 'I got a job with Amazon last year at the fulfillment center, had my set pay rate, schedule and everything, but due to COVID-19, they kept pushing the opening date back. Months later, no word from them no update or anything and in hindsight, it probably was for the best',
 'Why are the bosses trying to tell the employees what is better for them? The employee knows what is best for themselves and their families better than some “detached from reality” corporate eejit!',
 "Ya lazy people! Build ya own dang company if ya don't like Amazon's policies!",
 'There’s nothing the government can do. Any consequences would be negligible. Support striking workers.']

In [ ]:
print("\nTop 5 Neutral Comments:")
df[df["tb_label"] == "neutral"]["text"].head(5).tolist()


Top 5 Neutral Comments:


["I'm suspended for having my Medical Marijuana license.  In a state that recreational is legal and I have my medical card for this state. This was all retaliation by a safety employee that dislikes me and my medical rights. He said I have no medical  rights when at work.",
 'Amazon employees are over worked with mandatory overtime on already 12 hour per day shifts...so they want you to work on your off days...SUE his ass',
 'DONT BELIEVE THE HYPE! IF MANAGEMENT DONT GIVE U A VOICE, HOW CAN U BE HEARD! THATS THE JOB FOR UNIONS, TO GIVE U A VOICE!',
 "Correct me if I am wrong. From all the people around me talking, I see this practice,  and then I see police corruption, officers getting paid suspension, and lawsuits against corruption. But, taxpayers has to foot the bill, and then residents property taxes is raised along with other miscellaneous stuff (not including officials going on golf outings off of taxpayers expense) and rent being raised at an all time high, minimum wage doesn't 

In [ ]:
print("\nTop 5 Negative Comments:")
df[df["tb_label"] == "negative"]["text"].head(5).tolist()


Top 5 Negative Comments:


['My step son works at Amazon   It’s pretty terrible. They rotate shifts, if he works overnight the night before the schedule comes out, he could be scheduled to work day shift that day. Meaning a 20 hour work day that he finds out about on his drive home.',
 'Union is made for lazy worker',
 'I worked at 2 Amazon warehouses in Germany and after that bad experience, I try my best to buy what I need from local stores, or other websites and Amazon is always the last choice for me.\nWorkers are being treated like slaves there. One of the worst place to work.',
 "I feel for the workers. I can't imagine the pressure they endure. But man the technology and innovation behind this is an insane feat.",
 'All these customers fail to realize how much hard work goes into every step of the way and go full-on Karen mode if you\'re late. We\'ve had people scream at us about their package being delayed due to weather, offer them the option to come pick it up directly from the warehouse, only to be ans

In [1]:
# Labeling using Vader

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    return analyzer.polarity_scores(text)['compound']

vader_scores= df[comment_col].apply(get_vader_scores)

def score_to_label(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

df["vader_label"] = vader_scores.apply(score_to_label)

df["vader_label"].value_counts()

ModuleNotFoundError: No module named 'vaderSentiment'

In [ ]:
print("Top 5 Positive Comments:")
df[df["vader_label"] == "positive"]["text"].head(5).tolist()

Top 5 Positive Comments:


["Correct me if I am wrong. From all the people around me talking, I see this practice,  and then I see police corruption, officers getting paid suspension, and lawsuits against corruption. But, taxpayers has to foot the bill, and then residents property taxes is raised along with other miscellaneous stuff (not including officials going on golf outings off of taxpayers expense) and rent being raised at an all time high, minimum wage doesn't help the unemployed, but majority jobs offered doesn't pay a cost of living, and hearing in Tennessee there's a bill been passed (hearing) if you're homeless you're going to be arrested,  and if you feed homeless people you are going to be arrested. But, taxpayers money is being sent overseas to fund a war WE never created, and now Ukraine immigrants ( same people from video footage we don't control) shows the world they can't get on a subway/bus cause they are less human, while homeless veterans are living on the streets,  but these foreign people 

In [ ]:
print("\nTop 5 Neutral Comments:")
df[df["vader_label"] == "neutral"]["text"].head(5).tolist()


Top 5 Neutral Comments:


['DONT BELIEVE THE HYPE! IF MANAGEMENT DONT GIVE U A VOICE, HOW CAN U BE HEARD! THATS THE JOB FOR UNIONS, TO GIVE U A VOICE!',
 "They're talking about sub-same-day warehouses where the flex drivers do the delivering for those 4 hr delivery window orders.",
 'i looking job location',
 'I’m a delivery driver and these dumbasses can’t even put a sticker on or tape up a box correctly. They make my job harder everyday',
 "3:50 no, it's to divert liabilities from Amazon to the worker, who often is considered self employed."]

In [ ]:
print("\nTop 5 Negative Comments:")
df[df["vader_label"] == "negative"]["text"].head(5).tolist()


Top 5 Negative Comments:


["I'm suspended for having my Medical Marijuana license.  In a state that recreational is legal and I have my medical card for this state. This was all retaliation by a safety employee that dislikes me and my medical rights. He said I have no medical  rights when at work.",
 'Amazon employees are over worked with mandatory overtime on already 12 hour per day shifts...so they want you to work on your off days...SUE his ass',
 "Ya lazy people! Build ya own dang company if ya don't like Amazon's policies!",
 'So I gotta pay taxes?! And dont have a Right? To organize ?',
 "At some point, Amazon will have a tough time getting folks to work in their warehouses. Last year, here in Illinois, 6 people died when a tornado hit an Amazon warehouse downstate, and the workers said that they couldn't leave, even though they knew the tornado was coming. They said that they were threatened with immediate termination if they left, and went home. I guess that would've been better than staying there, and 

In [ ]:
# Checking Similarity in vader and textblob labels

agreement = df[df["tb_label"] == df["vader_label"]]
disagreement = df[df["tb_label"] != df["vader_label"]]

print("Disagreement count:", len(disagreement))

print("Agreement distribution:")
print(agreement["tb_label"].value_counts())



Disagreement count: 1524
Agreement distribution:
tb_label
positive    1071
negative     536
neutral      254
Name: count, dtype: int64


In [ ]:
# Selecting data for Manual Validation

agreement_sample = agreement.sample(100, random_state=6)
disagreement_sample = disagreement.sample(500, random_state=7)

manual_set = pd.concat([agreement_sample, disagreement_sample])
manual_set.to_csv("manual_dataset.csv", index=False)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving manual_labelled.csv to manual_labelled.csv


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# TextBlob vs Manual
tb_acc = accuracy_score(df["manual_label"], df["tb_label"])
tb_f1 = f1_score(df["manual_label"], df["tb_label"], average="macro")

print("TextBlob Accuracy:", tb_acc)
print("TextBlob Macro F1:", tb_f1)
print("\nTextBlob Report:\n")
print(classification_report(df["manual_label"], df["tb_label"]))

In [ ]:

# VADER vs Manual
vader_acc = accuracy_score(df["manual_label"], df["vader_label"])
vader_f1 = f1_score(df["manual_label"], df["vader_label"], average="macro")

print("VADER Accuracy:", vader_acc)
print("VADER Macro F1:", vader_f1)
print("\nVADER Report:\n")
print(classification_report(df["manual_label"], df["vader_label"]))

# Ignore everything from below - not using any of it

In [ ]:
# Using Vader Labels as the final label as it is more accurate
df["final_label"] = df["vader_label"]

# Overwriting select final labels with the manually validated labels
manual_map = manual_df.set_index("text")["manual_label"]
df["final_label"] = df["text"].map(manual_map).combine_first(df["final_label"])

In [ ]:
final_df = df[["text", "final_label"]].copy()
final_df.rename(columns={"final_label": "Label"}, inplace=True)
final_df.head()

,text,Label
0,I'm suspended for having my Medical Marijuana ...,negative
4,Amazon employees are over worked with mandator...,negative
5,DONT BELIEVE THE HYPE! IF MANAGEMENT DONT GIVE...,negative
6,Correct me if I am wrong. From all the people ...,neutral
9,Amazon treats its workers like dirt and then a...,negative


In [ ]:
print("Top 5 Positive Comments:")
final_df[final_df["Label"] == "positive"]["text"].head(5).tolist()

Top 5 Positive Comments:


["Ya lazy people! Build ya own dang company if ya don't like Amazon's policies!",
 'Amazon’s same day delivery system is a fascinating example of how logistics, data, and automation can work together at massive scale. What really stands out is how Amazon redesigned its entire fulfillment network from giant warehouses to smaller, strategically placed facilities just to cut minutes out of every step. It’s impressive but also a reminder of how much invisible complexity goes into the order now, get it today experience we take for granted. The blend of human labor and automation, the emphasis on proximity, and the pressure to move faster all raise interesting questions about the future of retail, work, and sustainability.',
 'Amazon is amazing. But, please be kind to the workers...ok?',
 "This is actually quite scary. This company is so good and so efficient that they could take over any business if they wanted to. The business I'm in ( I won't say which one).. is highly specialized and wor

In [ ]:
print("Top 5 Neutral Comments:")
final_df[final_df["Label"] == "neutral"]["text"].head(5).tolist()

Top 5 Neutral Comments:


["Correct me if I am wrong. From all the people around me talking, I see this practice,  and then I see police corruption, officers getting paid suspension, and lawsuits against corruption. But, taxpayers has to foot the bill, and then residents property taxes is raised along with other miscellaneous stuff (not including officials going on golf outings off of taxpayers expense) and rent being raised at an all time high, minimum wage doesn't help the unemployed, but majority jobs offered doesn't pay a cost of living, and hearing in Tennessee there's a bill been passed (hearing) if you're homeless you're going to be arrested,  and if you feed homeless people you are going to be arrested. But, taxpayers money is being sent overseas to fund a war WE never created, and now Ukraine immigrants ( same people from video footage we don't control) shows the world they can't get on a subway/bus cause they are less human, while homeless veterans are living on the streets,  but these foreign people 

In [ ]:
print("Top 5 Negative Comments:")
final_df[final_df["Label"] == "negative"]["text"].head(5).tolist()

Top 5 Negative Comments:


["I'm suspended for having my Medical Marijuana license.  In a state that recreational is legal and I have my medical card for this state. This was all retaliation by a safety employee that dislikes me and my medical rights. He said I have no medical  rights when at work.",
 'Amazon employees are over worked with mandatory overtime on already 12 hour per day shifts...so they want you to work on your off days...SUE his ass',
 'DONT BELIEVE THE HYPE! IF MANAGEMENT DONT GIVE U A VOICE, HOW CAN U BE HEARD! THATS THE JOB FOR UNIONS, TO GIVE U A VOICE!',
 'Amazon treats its workers like dirt and then acts surprised when the workers want to unionize.?    They are getting what they asked for with the tactics they have used.',
 'I got a job with Amazon last year at the fulfillment center, had my set pay rate, schedule and everything, but due to COVID-19, they kept pushing the opening date back. Months later, no word from them no update or anything and in hindsight, it probably was for the best'

In [ ]:
final_df['Label'].value_counts()

,count
Label,
positive,1598
negative,1238
neutral,549


In [ ]:
# Performing Test Train Split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    final_df,
    test_size=0.2,
    random_state=21,
    stratify=final_df["Label"]
)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

Train size: (2708, 2)
Test size: (677, 2)


In [ ]:
train_df["Label"].value_counts()

,count
Label,
positive,1278
negative,991
neutral,439


In [ ]:
test_df["Label"].value_counts()

,count
Label,
positive,320
negative,247
neutral,110


In [ ]:
train_df.to_csv("train_dataset.csv", index=False)
test_df.to_csv("test_dataset.csv", index=False)